# R1-C04 · Sensibilidad a meteorología futura climatológica — TEST 2023–2024 (V01)

**Objetivo:** comparar, de manera pareada, el PI-LSTM congelado con (A) meteorología futura observada retrospectivamente y (B1) meteorología futura sustituida por medianas climatológicas ajustadas **solo en TRAIN 1998–2019**.

**Diseño:** mismo modelo (SHA-256), escaladores, 34 predictores históricos, geometría, codificaciones temporales, tres referencias clear-sky, targets, máscaras y reconstrucción NOB. Se sustituyen únicamente las **nueve** variables meteorológicas futuras escaladas. NO se usa información de 2020–2024 para ajustar climatología. NO se llama a `.fit()` ni se reentrena.

**Limitación:** B1 no reproduce el rollout 2050 (sin retroalimentación sintética); mide sensibilidad condicionada a contexto histórico observado, no pronóstico operativo equitativo ni contribución causal individual de las variables.

**Entradas a subir a `/content/`:** `consol_tumbabiro_2026_total.csv`, `PI_LSTM_V2_4_V1_logDHI_partition_stride2.keras`, `scaler_hist_v24_v1.joblib`, `scaler_fut_v24_v1.joblib`, `model_metadata_v24_v1.json`, `final_canonical_metrics_v24_v1_raw.csv` (opcional: `test_chronological_non_overlapping_v24_v1.csv`).

**Salida:** `R1_C04_TEST_METEOROLOGICAL_CLIMATOLOGY_SENSITIVITY_V01.zip` con métricas por protocolo, series NOB pareadas, climatología TRAIN, auditoría del reemplazo y manifiesto de hashes.

## 1. Identidad de los artefactos y contrato canónico

In [ ]:
from pathlib import Path
import os,hashlib,glob,json
ROOT=Path('/content') if Path('/content').exists() else Path('/mnt/data')
os.chdir(ROOT)
def locate(stem,ext,required=True):
    matches=[p for p in ROOT.iterdir() if p.is_file() and p.suffix.lower()==ext and (p.stem==stem or p.stem.startswith(stem+'('))]
    exact=[p for p in matches if p.stem==stem]
    if exact: return exact[0]
    if len(matches)==1: return matches[0]
    if not required and not matches:return None
    raise FileNotFoundError(f'Archivo no localizado inequívocamente: {stem}{ext}; coincidencias: {[p.name for p in matches]}')
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
DATA=locate('consol_tumbabiro_2026_total','.csv')
MODEL=locate('PI_LSTM_V2_4_V1_logDHI_partition_stride2','.keras')
SC_H=locate('scaler_hist_v24_v1','.joblib')
SC_F=locate('scaler_fut_v24_v1','.joblib')
META=locate('model_metadata_v24_v1','.json')
MODEL_SHA='5aef4783a3dc0b37b98711ebb489d20458f6c5efd2308ca579909e4cc8724a4e'
assert sha(MODEL)==MODEL_SHA,'HASH DE MODELO DIFERENTE AL CANÓNICO'
for p in [DATA,MODEL,SC_H,SC_F,META]:print(p.name, p.stat().st_size)
print('Modelo SHA256 PASS:',sha(MODEL))

CANON_METRICS=locate('final_canonical_metrics_v24_v1_raw','.csv',required=True)
CANON_NOB=locate('test_chronological_non_overlapping_v24_v1','.csv',required=False)
print('Referencia:',CANON_METRICS, 'NOB:',CANON_NOB)

In [ ]:
# ============================================================
# 1. PI-LSTM V2.4 — CONFIGURACIÓN GENERAL
# ============================================================



import os
import gc
import json
import random
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    Concatenate,
    LayerNormalization,
    AdditiveAttention,
    TimeDistributed,
    Rescaling,
    Layer
)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)


# ============================================================
# REPRODUCIBILIDAD
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# CONFIGURACIÓN CANÓNICA PI-LSTM V2.4
# ============================================================

@dataclass(frozen=True)
class CFG:

    # --------------------------------------------------------
    # Archivos
    # --------------------------------------------------------

    data_path: str = "consol_tumbabiro_2026_total.csv"

    model_path: str = (
        "PI_LSTM_V2_4_logDHI_partition_stride2.keras"
    )

    scaler_hist_path: str = (
        "scaler_hist_v24.joblib"
    )

    scaler_fut_path: str = (
        "scaler_fut_v24.joblib"
    )

    metadata_path: str = (
        "model_metadata_v24.json"
    )

    metrics_path: str = (
        "test_metrics_v24.csv"
    )


    # --------------------------------------------------------
    # División temporal
    # --------------------------------------------------------

    train_end: str = "2019-12-31 23:30:00"
    val_end: str = "2022-12-31 23:30:00"


    # --------------------------------------------------------
    # Ventanas temporales
    # --------------------------------------------------------

    L: int = 72
    H: int = 24
    stride: int = 2

    batch_size: int = 32

    epochs: int = 25


    # --------------------------------------------------------
    # Optimización
    # --------------------------------------------------------

    lr: float = 3e-4

    patience_es: int = 8

    patience_rlr: int = 3

    reduce_lr_factor: float = 0.5

    min_lr: float = 1e-5


    # --------------------------------------------------------
    # Arquitectura
    # --------------------------------------------------------

    latent_enc: int = 96
    latent_dec: int = 96

    dropout: float = 0.10


    # --------------------------------------------------------
    # Umbrales físicos
    # --------------------------------------------------------

    thr_cs: float = 20.0

    thr_day_cosz: float = 0.05


    # --------------------------------------------------------
    # Límites de las salidas físicas
    # --------------------------------------------------------

    # Rama directa:
    # la auditoría histórica mostró max(kDNI) = 1.0
    kdni_max: float = 1.20

    # Rama difusa:
    # auditoría histórica sin clipping:
    # max(kDHI) = 7.918
    #
    # Se adopta 8.0 como límite físico-numérico superior.
    kdhi_max: float = 8.00


    # --------------------------------------------------------
    # Pesos de la función de pérdida
    # --------------------------------------------------------

    lambda_kdni: float = 0.40

    # En V2.4 esta ponderación se aplica a qDHI = log1p(kDHI)
    lambda_qdhi: float = 1.45

    lambda_ghi: float = 0.65

    lambda_ratio: float = 1.90

    lambda_energy: float = 0.27


# ============================================================
# INSTANCIAR CONFIGURACIÓN
# ============================================================

cfg = CFG()


# ============================================================
# PARÁMETROS DERIVADOS DE LA TRANSFORMACIÓN DIFUSA
# ============================================================

# qDHI = log(1 + kDHI)
#
# Si kDHI_max = 8:
#
# qDHI_max = log(9) ≈ 2.1972246

QDHI_MAX = float(np.log1p(cfg.kdhi_max))


# ============================================================
# RESUMEN DE CONFIGURACIÓN
# ============================================================

print("TensorFlow:", tf.__version__)

print("\n=== PI-LSTM V2.4 ===")

print("SEED          :", SEED)

print("L             :", cfg.L)
print("H             :", cfg.H)
print("stride        :", cfg.stride)

print("batch_size    :", cfg.batch_size)
print("epochs_max    :", cfg.epochs)

print("learning rate :", cfg.lr)

print("kDNI max      :", cfg.kdni_max)
print("kDHI max      :", cfg.kdhi_max)

print("qDHI max      :", QDHI_MAX)

print("\nArtefactos:")
print("  modelo      :", cfg.model_path)
print("  scaler hist :", cfg.scaler_hist_path)
print("  scaler fut  :", cfg.scaler_fut_path)
print("  metadata    :", cfg.metadata_path)
print("  métricas    :", cfg.metrics_path)

cfg

# ============================================================
# VERIFICACIÓN DEL ENTORNO DE CÓMPUTO
# ============================================================

print("\n=== DISPOSITIVOS DISPONIBLES ===")

print(
    "GPU detectadas:",
    tf.config.list_physical_devices("GPU")
)

In [ ]:
object.__setattr__(cfg,'data_path',str(DATA)); object.__setattr__(cfg,'model_path',str(MODEL))
print('Datos:',cfg.data_path,'Modelo:',cfg.model_path)


In [ ]:
# ============================================================
# 2. PI-LSTM V2.4 — COLUMNAS CANÓNICAS Y CARGA DEL HISTÓRICO
# ============================================================

# ------------------------------------------------------------
# Columnas temporales
# ------------------------------------------------------------

COL_DATE = "Date"
COL_HOUR = "Hour"
COL_MIN  = "Minute"


# ------------------------------------------------------------
# Variables meteorológicas
# ------------------------------------------------------------

COL_TEMP = "Temperature"
COL_DEW  = "Dew Point"
COL_RH   = "Relative Humidity"
COL_PRES = "Pressure"
COL_AOD  = "Aerosol Optical Depth"
COL_PW   = "Precipitable Water"
COL_WSPD = "Wind Speed"
COL_WDIR = "Wind Direction"


# ------------------------------------------------------------
# Variables radiativas
# ------------------------------------------------------------

COL_GHI = "GHI"
COL_DNI = "DNI"
COL_DHI = "DHI"

COL_THZ = "Solar Zenith Angle"

COL_CSGHI = "Clearsky GHI"
COL_CSDNI = "Clearsky DNI"
COL_CSDHI = "Clearsky DHI"


# ============================================================
# CONTRATO MÍNIMO DEL ARCHIVO HISTÓRICO
# ============================================================

REQUIRED_RAW_COLS = [
    COL_DATE,
    COL_HOUR,
    COL_MIN,

    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,
    COL_WDIR,

    COL_GHI,
    COL_DNI,
    COL_DHI,
    COL_THZ,

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# ============================================================
# VERIFICACIÓN DE RUTA
# ============================================================

data_path = Path(cfg.data_path)

print("Directorio de trabajo:")
print(" ", Path.cwd())

print("\nArchivo histórico esperado:")
print(" ", data_path.resolve())


if not data_path.exists():

    raise FileNotFoundError(
        "\nNo se encontró el archivo histórico requerido:\n"
        f"  {data_path.resolve()}\n\n"
        "Verifique que 'consol_tumbabiro_2026_total.csv' "
        "se encuentre en el directorio de trabajo actual "
        "o modifique cfg.data_path con la ruta correcta."
    )


# ============================================================
# CARGA DEL CSV
# ============================================================

df_raw = pd.read_csv(
    data_path,
    low_memory=False
)


# ============================================================
# AUDITORÍA BÁSICA DE ESTRUCTURA
# ============================================================

missing_raw_cols = [
    c
    for c in REQUIRED_RAW_COLS
    if c not in df_raw.columns
]

if missing_raw_cols:

    raise ValueError(
        "Faltan columnas obligatorias en el histórico:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_raw_cols
        )
    )


# ============================================================
# RESUMEN
# ============================================================

file_size_mb = (
    data_path.stat().st_size
    / (1024 ** 2)
)

print("\n✓ Archivo histórico encontrado")
print(
    f"✓ Tamaño archivo : {file_size_mb:.2f} MB"
)

print(
    f"✓ Dimensión raw  : {df_raw.shape}"
)

print(
    f"✓ Número columnas: {df_raw.shape[1]}"
)

print(
    f"✓ Columnas contractuales presentes: "
    f"{len(REQUIRED_RAW_COLS)} / {len(REQUIRED_RAW_COLS)}"
)

print("\nPrimeras columnas:")
print(
    df_raw.columns.tolist()
)

In [ ]:
# ============================================================
# 3. PI-LSTM V2.4 — TIMESTAMP E INGENIERÍA FÍSICA BÁSICA
# ============================================================


# ============================================================
# 1. CONSTRUCCIÓN ROBUSTA DEL TIMESTAMP
# ============================================================

# ============================================================
# 1. CONSTRUCCIÓN ROBUSTA DEL TIMESTAMP
# ============================================================

def build_timestamp(df):
    """
    Construye timestamp a partir de Date, Hour y Minute.

    El histórico consolidado utiliza Date en formato:
        DD/MM/YYYY

    Ejemplo:
        13/01/1998 = 13 de enero de 1998

    Se utiliza un formato explícito para evitar interpretaciones
    ambiguas de pandas.
    """

    out = df.copy()

    # --------------------------------------------------------
    # Fecha: formato explícito DD/MM/YYYY
    # --------------------------------------------------------

    date_raw = (
        out[COL_DATE]
        .astype(str)
        .str.strip()
    )

    date_parsed = pd.to_datetime(
        date_raw,
        format="%d/%m/%Y",
        errors="coerce"
    )

    # --------------------------------------------------------
    # Hora y minuto
    # --------------------------------------------------------

    hour = pd.to_numeric(
        out[COL_HOUR],
        errors="coerce"
    )

    minute = pd.to_numeric(
        out[COL_MIN],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Validaciones de hora y minuto
    # --------------------------------------------------------

    invalid_hour = (
        hour.isna()
        | (hour < 0)
        | (hour > 23)
    )

    invalid_minute = (
        minute.isna()
        | ~minute.isin([0, 30])
    )

    if invalid_hour.any():

        print(
            "\nPrimeros valores inválidos de Hour:"
        )

        print(
            out.loc[
                invalid_hour,
                [COL_DATE, COL_HOUR, COL_MIN]
            ]
            .head(20)
            .to_string(index=False)
        )

        raise ValueError(
            "La columna Hour contiene valores inválidos."
        )

    if invalid_minute.any():

        print(
            "\nPrimeros valores inválidos de Minute:"
        )

        print(
            out.loc[
                invalid_minute,
                [COL_DATE, COL_HOUR, COL_MIN]
            ]
            .head(20)
            .to_string(index=False)
        )

        raise ValueError(
            "La columna Minute contiene valores distintos "
            "de 0 o 30."
        )

    # --------------------------------------------------------
    # Timestamp final
    # --------------------------------------------------------

    out["timestamp"] = (
        date_parsed
        + pd.to_timedelta(
            hour,
            unit="h"
        )
        + pd.to_timedelta(
            minute,
            unit="m"
        )
    )

    return out


# ============================================================
# 2. FEATURES TEMPORALES Y GEOMETRÍA
# ============================================================

def add_time_features(df):

    out = df.copy()

    ts = out["timestamp"]

    # --------------------------------------------------------
    # Ciclo diario
    # --------------------------------------------------------

    hour_float = (
        ts.dt.hour
        + ts.dt.minute / 60.0
    )

    out["sin_hour"] = np.sin(
        2.0 * np.pi * hour_float / 24.0
    ).astype(np.float32)

    out["cos_hour"] = np.cos(
        2.0 * np.pi * hour_float / 24.0
    ).astype(np.float32)


    # --------------------------------------------------------
    # Ciclo anual
    # Se conserva la formulación V2.3: 365.25 días
    # --------------------------------------------------------

    doy = ts.dt.dayofyear.astype(float)

    out["sin_doy"] = np.sin(
        2.0 * np.pi * doy / 365.25
    ).astype(np.float32)

    out["cos_doy"] = np.cos(
        2.0 * np.pi * doy / 365.25
    ).astype(np.float32)


    # --------------------------------------------------------
    # Coseno cenital positivo
    # --------------------------------------------------------

    theta = np.deg2rad(
        pd.to_numeric(
            out[COL_THZ],
            errors="coerce"
        ).to_numpy(dtype=np.float32)
    )

    out["cosz_plus"] = np.maximum(
        np.cos(theta),
        0.0
    ).astype(np.float32)

    return out


# ============================================================
# 3. ÍNDICES CLEAR-SKY — V2.4
# ============================================================

def add_kspace_features(df, thr_cs):

    out = df.copy()

    # --------------------------------------------------------
    # Convertir a arrays numéricos
    # --------------------------------------------------------

    csghi = pd.to_numeric(
        out[COL_CSGHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    csdni = pd.to_numeric(
        out[COL_CSDNI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    csdhi = pd.to_numeric(
        out[COL_CSDHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    ghi = pd.to_numeric(
        out[COL_GHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    dni = pd.to_numeric(
        out[COL_DNI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    dhi = pd.to_numeric(
        out[COL_DHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)


    # --------------------------------------------------------
    # kGHI
    # Se conserva como feature auxiliar.
    # --------------------------------------------------------

    out["kGHI"] = np.where(
        csghi > thr_cs,
        ghi / csghi,
        0.0
    ).astype(np.float32)

    out["kGHI"] = np.clip(
        out["kGHI"],
        0.0,
        2.0
    ).astype(np.float32)


    # --------------------------------------------------------
    # kDNI
    # Auditoría histórica:
    # max observado = 1.0
    # límite de salida = 1.20
    # --------------------------------------------------------

    out["kDNI"] = np.where(
        csdni > thr_cs,
        dni / csdni,
        0.0
    ).astype(np.float32)

    # Protección numérica histórica.
    # No afecta los datos auditados.
    out["kDNI"] = np.clip(
        out["kDNI"],
        0.0,
        cfg.kdni_max
    ).astype(np.float32)


    # --------------------------------------------------------
    # kDHI RAW
    #
    # V2.4 elimina el antiguo clipping a 2.0.
    # La auditoría histórica mostró:
    #
    # max = 7.918
    # P95 = 4.778
    # P99 = 5.918
    #
    # kdhi_max = 8.0 actúa únicamente como protección
    # físico-numérica y no censura el histórico auditado.
    # --------------------------------------------------------

    kdhi_raw = np.where(
        csdhi > thr_cs,
        dhi / csdhi,
        0.0
    ).astype(np.float32)

    out["kDHI"] = np.clip(
        kdhi_raw,
        0.0,
        cfg.kdhi_max
    ).astype(np.float32)


    # --------------------------------------------------------
    # Transformación logarítmica difusa
    #
    # qDHI = log(1 + kDHI)
    # --------------------------------------------------------

    out["qDHI"] = np.log1p(
        out["kDHI"].to_numpy(dtype=np.float32)
    ).astype(np.float32)


    return out


# ============================================================
# 4. VARIABLES DE PARTICIÓN, DINÁMICA Y MEMORIA
# ============================================================

def add_partition_features(df, thr_cs):

    out = df.copy()

    eps = 1e-6


    # --------------------------------------------------------
    # Rampas radiativas
    # --------------------------------------------------------

    out["dGHI"] = (
        out[COL_GHI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )

    out["dDNI"] = (
        out[COL_DNI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )

    out["dDHI"] = (
        out[COL_DHI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Fracción difusa
    # --------------------------------------------------------

    out["diffuse_ratio"] = np.where(
        out[COL_GHI] > thr_cs,
        out[COL_DHI]
        / (out[COL_GHI] + eps),
        0.0
    ).astype(np.float32)

    out["diffuse_ratio"] = (
        out["diffuse_ratio"]
        .clip(0.0, 1.2)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Fracción directa horizontal
    # --------------------------------------------------------

    out["beam_ratio"] = np.where(
        out[COL_GHI] > thr_cs,
        (
            out[COL_DNI]
            * out["cosz_plus"]
        )
        / (out[COL_GHI] + eps),
        0.0
    ).astype(np.float32)

    out["beam_ratio"] = (
        out["beam_ratio"]
        .clip(0.0, 1.5)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Variables con lag 1
    #
    # Se conserva kDHI como feature física histórica.
    # Se añade qDHI para disponer también de memoria en
    # el espacio transformado.
    # --------------------------------------------------------

    lag_bases = [
        COL_DNI,
        COL_DHI,
        COL_GHI,

        "kGHI",
        "kDNI",
        "kDHI",
        "qDHI",

        "diffuse_ratio",
        "beam_ratio",
    ]

    for base in lag_bases:

        out[f"{base}_lag1"] = (
            out[base]
            .shift(1)
        )


    # --------------------------------------------------------
    # Promedios móviles de tres pasos previos
    # 3 × 30 min = 90 min
    # --------------------------------------------------------

    roll_bases = [
        COL_DNI,
        COL_DHI,

        "kGHI",
        "kDNI",
        "kDHI",
        "qDHI",

        "diffuse_ratio",
        "beam_ratio",
    ]

    for base in roll_bases:

        out[f"{base}_roll3"] = (
            out[base]
            .shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )


    # --------------------------------------------------------
    # Dirección del viento como variable circular
    # --------------------------------------------------------

    wdir = np.deg2rad(
        pd.to_numeric(
            out[COL_WDIR],
            errors="coerce"
        ).fillna(0.0)
    )

    out["sin_wdir"] = np.sin(
        wdir
    ).astype(np.float32)

    out["cos_wdir"] = np.cos(
        wdir
    ).astype(np.float32)


    # --------------------------------------------------------
    # En esta etapa únicamente los NaN producidos por lag/roll
    # iniciales se sustituyen por cero, manteniendo la lógica
    # original de V2.3.
    # --------------------------------------------------------

    return out.fillna(0.0)


# ============================================================
# 5. CONSTRUIR DATAFRAME TEMPORAL
# ============================================================

df = build_timestamp(df_raw)


# ------------------------------------------------------------
# Auditoría del timestamp antes de eliminar registros
# ------------------------------------------------------------

n_bad_timestamp = int(
    df["timestamp"].isna().sum()
)

print(
    "Timestamps inválidos:",
    n_bad_timestamp
)

if n_bad_timestamp > 0:

    print(
        "\nPrimeros registros con timestamp inválido:"
    )

    print(
        df.loc[
            df["timestamp"].isna(),
            [COL_DATE, COL_HOUR, COL_MIN]
        ]
        .head(20)
        .to_string(index=False)
    )

    raise ValueError(
        "Existen timestamps inválidos. "
        "Debe revisarse el formato de Date antes de continuar."
    )


# ------------------------------------------------------------
# Orden y unicidad
# ------------------------------------------------------------

df = (
    df
    .sort_values("timestamp")
    .reset_index(drop=True)
)


n_duplicates = int(
    df["timestamp"].duplicated().sum()
)

print(
    "Timestamps duplicados:",
    n_duplicates
)

if n_duplicates > 0:

    duplicated_rows = df.loc[
        df["timestamp"].duplicated(
            keep=False
        ),
        [
            "timestamp",
            COL_DATE,
            COL_HOUR,
            COL_MIN
        ]
    ]

    print(
        duplicated_rows
        .head(30)
        .to_string(index=False)
    )

    raise ValueError(
        "Se encontraron timestamps duplicados. "
        "No se eliminarán automáticamente."
    )


# ============================================================
# 6. AUDITORÍA ESTRICTA DE CONTINUIDAD A 30 MIN
# ============================================================

dt_min = (
    df["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60.0)
)

bad_dt = (
    dt_min.notna()
    & ~np.isclose(
        dt_min,
        30.0
    )
)

n_bad_dt = int(
    bad_dt.sum()
)

print(
    "Intervalos distintos de 30 min:",
    n_bad_dt
)


if n_bad_dt > 0:

    bad_idx = np.where(
        bad_dt.to_numpy()
    )[0]

    gaps = []

    for i in bad_idx[:50]:

        gaps.append({
            "index": int(i),

            "timestamp_anterior":
                df.loc[
                    i - 1,
                    "timestamp"
                ],

            "timestamp_actual":
                df.loc[
                    i,
                    "timestamp"
                ],

            "delta_min":
                float(
                    dt_min.iloc[i]
                ),
        })

    gaps_df = pd.DataFrame(gaps)

    print(
        "\n=== PRIMEROS INTERVALOS ANÓMALOS ==="
    )

    print(
        gaps_df.to_string(
            index=False
        )
    )

    print(
        "\nDistribución de intervalos:"
    )

    print(
        dt_min
        .dropna()
        .value_counts()
        .sort_index()
        .head(30)
    )

    raise ValueError(
        "El histórico no es continuo a 30 min. "
        "Revise los intervalos mostrados antes de continuar."
    )


print(
    "✓ Histórico continuo a 30 min."
)


# ============================================================
# 7. CONSTRUIR FEATURES
# ============================================================

df = add_time_features(df)

df = add_kspace_features(
    df,
    cfg.thr_cs
)

df = add_partition_features(
    df,
    cfg.thr_cs
)


# ============================================================
# 8. VALIDACIONES NUMÉRICAS BÁSICAS
# ============================================================

required_engineered = [
    "timestamp",
    "cosz_plus",

    "sin_hour",
    "cos_hour",
    "sin_doy",
    "cos_doy",

    "kGHI",
    "kDNI",
    "kDHI",
    "qDHI",

    "diffuse_ratio",
    "beam_ratio",

    "sin_wdir",
    "cos_wdir",
]


missing_engineered = [
    c
    for c in required_engineered
    if c not in df.columns
]

if missing_engineered:

    raise ValueError(
        "Faltan variables de ingeniería:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_engineered
        )
    )


# ------------------------------------------------------------
# Comprobaciones del dominio
# ------------------------------------------------------------

if (
    df["kDNI"].min() < 0.0
    or
    df["kDNI"].max()
    > cfg.kdni_max + 1e-6
):
    raise ValueError(
        "kDNI fuera del dominio configurado."
    )


if (
    df["kDHI"].min() < 0.0
    or
    df["kDHI"].max()
    > cfg.kdhi_max + 1e-6
):
    raise ValueError(
        "kDHI fuera del dominio configurado."
    )


if (
    df["qDHI"].min() < 0.0
    or
    df["qDHI"].max()
    > QDHI_MAX + 1e-6
):
    raise ValueError(
        "qDHI fuera del dominio configurado."
    )


# ============================================================
# 9. RESUMEN
# ============================================================

print("\n=== FEATURE DATAFRAME V2.4 ===")

print(
    "Dimensión:",
    df.shape
)

print(
    "Rango temporal:",
    df["timestamp"].min(),
    "->",
    df["timestamp"].max()
)

print(
    "kDNI min/max:",
    float(df["kDNI"].min()),
    float(df["kDNI"].max())
)

print(
    "kDHI min/max:",
    float(df["kDHI"].min()),
    float(df["kDHI"].max())
)

print(
    "qDHI min/max:",
    float(df["qDHI"].min()),
    float(df["qDHI"].max())
)

print(
    "QDHI_MAX teórico:",
    QDHI_MAX
)

print(
    "\n✓ Celda 3 completada correctamente."
)

In [ ]:
# ============================================================
# 4. PI-LSTM V2.4 — CONTRATO DE FEATURES Y TARGETS
# ============================================================


# ============================================================
# 1. VARIABLES HISTÓRICAS
#
# Se conserva el contrato dimensional:
#
#     HIST_COLS = 34
#
# Respecto de V2.3:
#
#   kDHI_lag1   -> qDHI_lag1
#   kDHI_roll3  -> qDHI_roll3
#
# kDHI instantáneo se conserva como predictor físico.
# ============================================================

HIST_COLS = [

    # --------------------------------------------------------
    # Persistencia y dinámica radiativa
    # --------------------------------------------------------

    "DNI_lag1",
    "DNI_roll3",

    "DHI_lag1",
    "DHI_roll3",

    "GHI_lag1",

    "dGHI",
    "dDNI",
    "dDHI",


    # --------------------------------------------------------
    # Índices clear-sky instantáneos
    # --------------------------------------------------------

    "kGHI",
    "kDNI",
    "kDHI",


    # --------------------------------------------------------
    # Memoria de índices clear-sky
    # --------------------------------------------------------

    "kGHI_lag1",
    "kGHI_roll3",

    "kDNI_lag1",
    "kDNI_roll3",

    # V2.4:
    # memoria difusa en espacio logarítmico
    "qDHI_lag1",
    "qDHI_roll3",


    # --------------------------------------------------------
    # Memoria de partición radiativa
    # --------------------------------------------------------

    "diffuse_ratio_lag1",
    "diffuse_ratio_roll3",

    "beam_ratio_lag1",
    "beam_ratio_roll3",


    # --------------------------------------------------------
    # Meteorología
    # --------------------------------------------------------

    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,

    "sin_wdir",
    "cos_wdir",


    # --------------------------------------------------------
    # Geometría y clear-sky
    # --------------------------------------------------------

    "cosz_plus",

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# ============================================================
# 2. VARIABLES FUTURAS ESCALADAS
# ============================================================

FUT_COLS_SCALED = [

    # Geometría
    "cosz_plus",

    # Tiempo cíclico
    "sin_hour",
    "cos_hour",
    "sin_doy",
    "cos_doy",

    # Meteorología
    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,

    # Dirección del viento circular
    "sin_wdir",
    "cos_wdir",
]


# ============================================================
# 3. VARIABLES FUTURAS FÍSICAS SIN ESCALAR
# ============================================================

FUT_COLS_RAW = [

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# Contrato futuro completo

FUT_COLS = (
    FUT_COLS_SCALED
    + FUT_COLS_RAW
)


# ============================================================
# 4. TARGETS DEL MODELO
#
# V2.4:
#
#   salida 1 -> kDNI
#   salida 2 -> qDHI = log(1 + kDHI)
#
# Además se conservan variables físicas auxiliares en Y_EXT
# para construir la loss físicamente informada.
# ============================================================

TARGET_COLS = [
    "kDNI",
    "qDHI",
]


# ============================================================
# 5. TARGET EXTENDIDO PARA LA LOSS
#
# Orden contractual:
#
# 0 -> kDNI_true
# 1 -> qDHI_true
# 2 -> GHI_true
# 3 -> DNI_cs
# 4 -> DHI_cs
# 5 -> GHI_cs
# 6 -> cosz_plus
# 7 -> DHI_true
#
# DHI_true se incorpora explícitamente en V2.4 para facilitar
# diagnósticos y futuras extensiones de la loss.
# ============================================================

Y_EXT_COLS = [

    "kDNI",
    "qDHI",

    COL_GHI,

    COL_CSDNI,
    COL_CSDHI,
    COL_CSGHI,

    "cosz_plus",

    COL_DHI,
]


# ============================================================
# 6. VALIDACIÓN DEL CONTRATO
# ============================================================

print("=== CONTRATO DE VARIABLES PI-LSTM V2.4 ===")

print(
    "HIST_COLS       :",
    len(HIST_COLS)
)

print(
    "FUT scaled      :",
    len(FUT_COLS_SCALED)
)

print(
    "FUT raw         :",
    len(FUT_COLS_RAW)
)

print(
    "FUT total       :",
    len(FUT_COLS)
)

print(
    "TARGET_COLS     :",
    len(TARGET_COLS)
)

print(
    "Y_EXT_COLS      :",
    len(Y_EXT_COLS)
)


# ------------------------------------------------------------
# Contratos dimensionales
# ------------------------------------------------------------

assert len(HIST_COLS) == 34, (
    f"Se esperaban 34 variables históricas "
    f"y se obtuvieron {len(HIST_COLS)}."
)

assert len(FUT_COLS_SCALED) == 14, (
    f"Se esperaban 14 variables futuras escaladas "
    f"y se obtuvieron {len(FUT_COLS_SCALED)}."
)

assert len(FUT_COLS_RAW) == 3

assert len(FUT_COLS) == 17

assert len(TARGET_COLS) == 2

assert len(Y_EXT_COLS) == 8


# ============================================================
# 7. VERIFICAR EXISTENCIA DE TODAS LAS COLUMNAS
# ============================================================

all_required_model_cols = list(dict.fromkeys(
    HIST_COLS
    + FUT_COLS
    + TARGET_COLS
    + Y_EXT_COLS
))


missing_model_cols = [
    c
    for c in all_required_model_cols
    if c not in df.columns
]


if missing_model_cols:

    raise ValueError(
        "Faltan columnas requeridas por el contrato V2.4:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_model_cols
        )
    )


# ============================================================
# 8. VERIFICACIÓN DE FINITUD
# ============================================================

nonfinite_report = {}

for col in all_required_model_cols:

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    n_bad = int(
        np.sum(
            ~np.isfinite(values)
        )
    )

    if n_bad > 0:
        nonfinite_report[col] = n_bad


if nonfinite_report:

    raise ValueError(
        "Se encontraron valores no finitos en variables "
        "del contrato:\n"
        + json.dumps(
            nonfinite_report,
            indent=2
        )
    )


# ============================================================
# 9. VERIFICACIONES ESPECÍFICAS DE V2.4
# ============================================================

assert np.all(
    df["qDHI"].to_numpy(dtype=float)
    >= 0.0
)

assert (
    float(df["qDHI"].max())
    <= QDHI_MAX + 1e-6
)

# Verificar consistencia exacta:
#
# qDHI = log1p(kDHI)

qdhi_reconstructed = np.log1p(
    df["kDHI"].to_numpy(dtype=float)
)

qdhi_error = np.max(
    np.abs(
        qdhi_reconstructed
        - df["qDHI"].to_numpy(dtype=float)
    )
)

print(
    "\nMáximo error qDHI - log1p(kDHI):",
    qdhi_error
)

assert qdhi_error < 1e-5


# ============================================================
# 10. RESUMEN DEL CONTRATO
# ============================================================

print("\n✓ Contrato V2.4 válido")

print(
    "✓ Input histórico :",
    f"{cfg.L} × {len(HIST_COLS)}"
)

print(
    "✓ Input futuro    :",
    f"{cfg.H} × {len(FUT_COLS)}"
)

print(
    "✓ Target neuronal :",
    f"{cfg.H} × {len(TARGET_COLS)}"
)

print(
    "✓ Target extendido:",
    f"{cfg.H} × {len(Y_EXT_COLS)}"
)

print("\nTargets neuronales:")
print(TARGET_COLS)

print("\nMemoria difusa V2.4:")
print(
    [
        "kDHI",
        "qDHI_lag1",
        "qDHI_roll3"
    ]
)

In [ ]:
import joblib
meta=json.loads(META.read_text(encoding='utf-8'))
sh=joblib.load(SC_H);sf=joblib.load(SC_F)
assert sh.n_features_in_==34 and sf.n_features_in_==14
assert list(sh.feature_names_in_)==list(HIST_COLS),'Orden histórico incongruente'
assert list(sf.feature_names_in_)==list(FUT_COLS_SCALED),'Orden futuro incongruente'
assert np.isfinite(sh.scale_).all() and np.isfinite(sf.scale_).all()
assert (sh.scale_>0).all() and (sf.scale_>0).all()
train_end=pd.Timestamp(cfg.train_end);val_end=pd.Timestamp(cfg.val_end)
train_df=df.loc[df.timestamp<=train_end].copy().reset_index(drop=True)
val_df=df.loc[(df.timestamp>train_end)&(df.timestamp<=val_end)].copy().reset_index(drop=True)
test_df=df.loc[df.timestamp>val_end].copy().reset_index(drop=True)
assert (len(train_df),len(val_df),len(test_df))==(385680,52608,35088)
assert (test_df.timestamp.diff().dropna()==pd.Timedelta(minutes=30)).all()
assert (test_df.timestamp.dt.date==pd.Timestamp('2024-02-29').date()).sum()==48
# Transformar con objetos congelados: ¡nunca fit()!
XH=sh.transform(test_df[HIST_COLS]).astype('float32')
XF=np.concatenate([sf.transform(test_df[FUT_COLS_SCALED]).astype('float32'),test_df[FUT_COLS_RAW].to_numpy(dtype='float32')],axis=1)
YT=test_df[Y_EXT_COLS].to_numpy(dtype='float32')
assert XH.shape==(35088,34) and XF.shape==(35088,17) and YT.shape==(35088,8)
N=(len(test_df)-cfg.L-cfg.H)//cfg.stride+1
assert N==17497
print('PASS: TRAIN VAL TEST',len(train_df),len(val_df),len(test_df),'ventanas:',N)


In [ ]:
# Cargar modelo sin compilar ni entrenar; soportar capas serializadas del notebook original.
# Algunas versiones de Keras requieren registrar la capa Expm1Layer.
import tensorflow as tf
from tensorflow.keras.utils import register_keras_serializable
@register_keras_serializable(package='Custom')
class Expm1Layer(tf.keras.layers.Layer):
 def call(self,inputs):return tf.math.expm1(inputs)
 def get_config(self):return super().get_config()
try:
 model=tf.keras.models.load_model(MODEL,compile=False,custom_objects={'Expm1Layer':Expm1Layer},safe_mode=True)
except Exception as exc:
 raise RuntimeError('No se pudo cargar el modelo congelado con seguridad. Comparta el traceback; no desactive safe_mode ni altere el modelo.') from exc
assert model.input_shape[0][-2:]==(72,34) and model.input_shape[1][-2:]==(24,17)
assert model.output_shape[-2:]==(24,2)
assert model.count_params()==110146
print('PASS:',model.input_shape,model.output_shape,model.count_params(),'TF',tf.__version__)


## 2. Climatología B1: únicamente TRAIN 1998–2019
Se replica el algoritmo de agrupación del notebook 2050 (día climatológico sin 29 de febrero, franja semihoraria, medianas de siete magnitudes y medianas independientes de `sin_wdir` / `cos_wdir`), pero sin incluir VALIDATION ni TEST. Para 29 de febrero de 2024 se interpola determinísticamente entre 28 de febrero y 1 de marzo, **solo para ese día**, ya que 2050 no es bisiesto.

In [ ]:
# Contrato explícito de sustitución: SOLO estos 9 predictores futuros.
METEO_FUT = [COL_TEMP,COL_DEW,COL_RH,COL_PRES,COL_AOD,COL_PW,COL_WSPD,'sin_wdir','cos_wdir']
assert len(METEO_FUT)==9 and len(set(METEO_FUT))==9
assert all(x in FUT_COLS_SCALED for x in METEO_FUT)
KEEP_FUT=[x for x in FUT_COLS if x not in METEO_FUT]
assert len(KEEP_FUT)==8 and {'cosz_plus','sin_hour','cos_hour','sin_doy','cos_doy'}.issubset(KEEP_FUT)
assert set(FUT_COLS_RAW).issubset(KEEP_FUT)
assert train_df.timestamp.max()==pd.Timestamp('2019-12-31 23:30:00')
assert test_df.timestamp.min()==pd.Timestamp('2023-01-01 00:00:00')
assert len(train_df)==385680

train_clim=train_df.copy()
assert 'sin_wdir' in train_clim and 'cos_wdir' in train_clim
train_clim=train_clim[~((train_clim.timestamp.dt.month==2)&(train_clim.timestamp.dt.day==29))].copy()
train_clim['month_day']=train_clim.timestamp.dt.strftime('%m-%d')
train_clim['slot']=(train_clim.timestamp.dt.hour*2+train_clim.timestamp.dt.minute//30).astype(int)
# Climatología original 2050: mediana de sin/cos, NO mediana del ángulo.
PROFILE=(train_clim.groupby(['month_day','slot'],sort=True)[METEO_FUT].median().sort_index())
COUNTS=train_clim.groupby(['month_day','slot']).size()
assert len(PROFILE)==365*48 and int(COUNTS.min())==22 and int(COUNTS.max())==22
assert PROFILE.notna().all().all()
assert '02-29' not in PROFILE.index.get_level_values('month_day')
assert train_clim.timestamp.max()<test_df.timestamp.min()

# Emparejar climatología a cada instante TEST; el 29-Feb se interpola sin usar sus observaciones.
test_keys=test_df.timestamp.dt.strftime('%m-%d')
test_slots=(test_df.timestamp.dt.hour*2+test_df.timestamp.dt.minute//30).astype(int)
query=pd.MultiIndex.from_arrays([test_keys,test_slots],names=['month_day','slot'])
B_METEO=PROFILE.reindex(query).reset_index(drop=True)
leap=(test_keys=='02-29').to_numpy()
assert int(leap.sum())==48
for slot in range(48):
    indices=np.flatnonzero(leap & (test_slots.to_numpy()==slot))
    assert len(indices)==1
    # Los dos perfiles proceden de TRAIN exclusivamente.
    B_METEO.iloc[indices[0]]=0.5*(PROFILE.loc[('02-28',slot)]+PROFILE.loc[('03-01',slot)])
assert B_METEO.notna().all().all() and np.isfinite(B_METEO.to_numpy(dtype=float)).all()
B_METEO.columns=METEO_FUT
B_METEO=B_METEO.astype('float64')
print('Climatología TRAIN:',PROFILE.shape,'observaciones por celda:',COUNTS.min(),COUNTS.max())
print('Filas TEST con climatología:',len(B_METEO),'29-feb interpoladas:',int(leap.sum()))

## 3. Matrices A y B1 congeladas y auditoría de no modificación

In [ ]:
# Partimos de df TEST físicamente procesado en el contrato original.
# En B1, transformar con el mismo scaler ajustado en TRAIN; JAMÁS re-ajustar.
FUT_B=test_df[FUT_COLS_SCALED].copy()
FUT_B.loc[:,METEO_FUT]=B_METEO[METEO_FUT].to_numpy(dtype='float64')
XFB=np.concatenate([
    sf.transform(FUT_B[FUT_COLS_SCALED]).astype('float32'),
    test_df[FUT_COLS_RAW].to_numpy(dtype='float32')
],axis=1)
assert XFB.shape==XF.shape==(35088,17)
assert np.isfinite(XFB).all() and np.isfinite(XF).all()
assert np.array_equal(XFB[:,[FUT_COLS.index(x) for x in KEEP_FUT]],XF[:,[FUT_COLS.index(x) for x in KEEP_FUT]])
assert np.array_equal(sh.transform(test_df[HIST_COLS]).astype('float32'),XH)
changed=(XFB!=XF)
assert not changed[:,[FUT_COLS.index(x) for x in KEEP_FUT]].any()
change_per_col=pd.DataFrame([{
 'feature':col,'group':'replaced_B1' if col in METEO_FUT else 'unchanged',
 'different_rows':int(changed[:,j].sum()),
 'max_abs_delta_scaled':float(np.abs(XFB[:,j]-XF[:,j]).max())
} for j,col in enumerate(FUT_COLS)])
assert (change_per_col.query("group=='replaced_B1'").different_rows>0).all()
assert (change_per_col.query("group=='unchanged'").different_rows==0).all()
scaler_sha_before={'historical':sha(SC_H),'future':sha(SC_F)}
model_sha_before=sha(MODEL)
print(change_per_col.to_string(index=False))
print('PASS: las 8 variables futuras restantes y los 34 predictores históricos se mantienen idénticos.')

## 4. Inferencia pareada (sin entrenamiento)
Se evalúan ambas configuraciones en **el mismo lote** para evitar diferencias por tamaño de batch; se preserva el orden original de las 17 497 ventanas.

In [ ]:
N=(len(test_df)-cfg.L-cfg.H)//cfg.stride+1
assert N==17497
BATCH=64
PA=np.empty((N,cfg.H,2),dtype='float32')
PB=np.empty((N,cfg.H,2),dtype='float32')
for a in range(0,N,BATCH):
    ids=np.arange(a,min(a+BATCH,N))
    st=ids*cfg.stride;en=st+cfg.L
    hx=np.stack([XH[s:s+cfg.L] for s in st])
    fa=np.stack([XF[e:e+cfg.H] for e in en])
    fb=np.stack([XFB[e:e+cfg.H] for e in en])
    PA[a:a+len(ids)]=model([hx,fa],training=False).numpy()
    PB[a:a+len(ids)]=model([hx,fb],training=False).numpy()
    if a==0 or (a//BATCH)%40==0:print('Ventanas:',min(a+BATCH,N),'/',N)
assert np.isfinite(PA).all() and np.isfinite(PB).all()
for arr in (PA,PB):
 assert arr[:,:,0].min()>=-1e-5 and arr[:,:,0].max()<=1.20001
 assert arr[:,:,1].min()>=-1e-5 and arr[:,:,1].max()<=np.log1p(8)+1e-5
assert sha(MODEL)==model_sha_before==MODEL_SHA
assert scaler_sha_before=={'historical':sha(SC_H),'future':sha(SC_F)}
print('PASS: inferencia B1 y A, sin cambio de pesos ni scalers.')

## 5. Reconstrucción y métricas pareadas
OVERLAP se calcula sobre todos los pares; BY_LEAD separa cada horizonte; NOB selecciona una predicción por timestamp. En NOB/DAYTIME se utiliza el filtro canónico `cosz_plus > 0.05`, como en las métricas históricas. También se exporta `DAYTIME_CS20` como diagnóstico alternativo, sin confundirlo con el canónico.

In [ ]:
from sklearn.metrics import r2_score
origins=np.arange(N)*cfg.stride+cfg.L
indices=origins[:,None]+np.arange(cfg.H)
aux=np.stack([YT[o:o+cfg.H] for o in origins])
cosz=aux[:,:,6];dni_cs=aux[:,:,3];dhi_cs=aux[:,:,4];ghi_cs=aux[:,:,5]
truth=np.stack([
 aux[:,:,2],
 test_df['DNI'].to_numpy(dtype='float32')[indices],
 aux[:,:,7]
],axis=-1)
def reconstruct(z):
 dni=z[:,:,0]*dni_cs
 dhi=np.expm1(z[:,:,1])*dhi_cs
 ghi=dni*cosz+dhi
 return np.stack([ghi,dni,dhi],axis=-1)
PRED={'OBS_METEO':reconstruct(PA),'TRAIN_CLIM_METEO':reconstruct(PB)}
for pred in PRED.values():
 assert np.isfinite(pred).all() and pred.shape==truth.shape==(17497,24,3)
 assert float(np.max(np.abs(pred[:,:,0]-(pred[:,:,1]*cosz+pred[:,:,2]))))<1e-3
mask_d=cosz>cfg.thr_day_cosz
mask_dc=mask_d & (ghi_cs>cfg.thr_cs)
mask_all=np.ones_like(mask_d,dtype=bool)
sel=np.arange(0,N,cfg.H//cfg.stride)
assert len(sel)==1459
nob_timestamps=test_df.timestamp.to_numpy()[indices[sel]].reshape(-1)
assert len(nob_timestamps)==35016 and len(np.unique(nob_timestamps))==35016
assert (pd.Series(nob_timestamps).diff().dropna()==pd.Timedelta(minutes=30)).all()

def stats(ob,pr):
 ob=np.asarray(ob,dtype='float64').reshape(-1);pr=np.asarray(pr,dtype='float64').reshape(-1)
 assert len(ob)>1 and np.isfinite(ob).all() and np.isfinite(pr).all()
 err=pr-ob
 return {'N':len(ob),'RMSE':float(np.sqrt(np.mean(err*err))),
         'MAE':float(np.mean(np.abs(err))),'Bias':float(np.mean(err)),
         'R2':float(r2_score(ob,pr))}
VARS=('GHI','DNI','DHI')
ALL_METRICS=[]; LEAD_METRICS=[]; NOB_METRICS=[]; ENERGY=[]; PAIRED=[]
for scenario,pred in PRED.items():
 for k,v in enumerate(VARS):
  ob=truth[:,:,k];pr=pred[:,:,k]
  for domain,ma in [('GLOBAL',mask_all),('DAYTIME',mask_d),('DAYTIME_CS20',mask_dc)]:
   ALL_METRICS.append({'scenario':scenario,'variable':v,'domain':domain,'protocol':'OVERLAP',**stats(ob[ma],pr[ma])})
  for j in range(cfg.H):
   for domain,ma in [('GLOBAL',mask_all[:,j]),('DAYTIME',mask_d[:,j])]:
    LEAD_METRICS.append({'scenario':scenario,'variable':v,'domain':domain,'protocol':'BY_LEAD','lead':j+1,**stats(ob[ma,j],pr[ma,j])})
  for domain,ma in [('GLOBAL',mask_all[sel].reshape(-1)),('DAYTIME',mask_d[sel].reshape(-1)),('DAYTIME_CS20',mask_dc[sel].reshape(-1))]:
   NOB_METRICS.append({'scenario':scenario,'variable':v,'domain':domain,'protocol':'NOB',**stats(ob[sel].reshape(-1)[ma],pr[sel].reshape(-1)[ma])})
  eo=float(ob[sel].astype('float64').sum()*.5/1000)
  ep=float(pr[sel].astype('float64').sum()*.5/1000)
  ENERGY.append({'scenario':scenario,'variable':v,'N':35016,'observed_kWh_m2':eo,'predicted_kWh_m2':ep,'energy_bias_pct':100*(ep/eo-1)})
ALL_METRICS=pd.DataFrame(ALL_METRICS);LEAD_METRICS=pd.DataFrame(LEAD_METRICS)
NOB_METRICS=pd.DataFrame(NOB_METRICS);ENERGY=pd.DataFrame(ENERGY)
assert (len(ALL_METRICS),len(LEAD_METRICS),len(NOB_METRICS),len(ENERGY))==(18,288,18,6)

def paired_delta(frame,keys):
 a=frame[frame.scenario=='OBS_METEO'].drop(columns='scenario').set_index(keys)
 b=frame[frame.scenario=='TRAIN_CLIM_METEO'].drop(columns='scenario').set_index(keys)
 assert a.index.is_unique and b.index.is_unique and a.index.equals(b.index)
 res=a[['N','RMSE','MAE','Bias','R2']].add_suffix('_OBS').join(b[['N','RMSE','MAE','Bias','R2']].add_suffix('_CLIM'))
 assert (res.N_OBS==res.N_CLIM).all()
 for c in ('RMSE','MAE','Bias','R2'):
  res['delta_'+c]=res[c+'_CLIM']-res[c+'_OBS']
 res['delta_RMSE_pct']=100*res.delta_RMSE/res.RMSE_OBS
 return res.reset_index()
OVERLAP_DELTA=paired_delta(ALL_METRICS,['variable','domain','protocol'])
LEAD_DELTA=paired_delta(LEAD_METRICS,['variable','domain','protocol','lead'])
NOB_DELTA=paired_delta(NOB_METRICS,['variable','domain','protocol'])
print('NOB GLOBAL delta:',NOB_DELTA.query("domain=='GLOBAL'").to_string(index=False))
print('Máscaras DAYTIME distintas (pares):',int(np.sum(mask_d!=mask_dc)))

## 6. Conciliación canónica obligatoria y equivalencia temporal
Si las seis métricas canónicas NOB de A no coinciden dentro de tolerancias numéricas explícitas, la ejecución se detiene y NO emite un ZIP como resultado aceptado.

In [ ]:
reference=pd.read_csv(CANON_METRICS).rename(columns={'scope':'domain','RMSE_Wm2':'RMSE','MAE_Wm2':'MAE','Bias_Wm2':'Bias'})
reference['domain']=reference.domain.str.upper()
assert {'variable','domain','RMSE','MAE','Bias','R2'}.issubset(reference.columns)
verify=NOB_METRICS[(NOB_METRICS.scenario=='OBS_METEO') & NOB_METRICS.domain.isin(['GLOBAL','DAYTIME'])].merge(
 reference,on=['variable','domain'],suffixes=('_actual','_canonica'),validate='1:1')
assert len(verify)==6
CANON_DELTAS={}
for metric in ('RMSE','MAE','Bias','R2'):
 delta=float(np.max(np.abs(verify[metric+'_actual']-verify[metric+'_canonica'])))
 CANON_DELTAS[metric]=delta
 assert delta<(0.02 if metric!='R2' else 1e-4), f'Conciliación canónica falló: {metric}: {delta}'
print('PASS conciliación canónica:',CANON_DELTAS)
if CANON_NOB is not None:
 old=pd.read_csv(CANON_NOB,parse_dates=['timestamp'])
 assert len(old)==35016 and np.array_equal(old.timestamp.to_numpy(),pd.to_datetime(nob_timestamps).to_numpy())
 for j,v in enumerate(VARS):
  assert float(np.max(np.abs(old[f'{v}_pred'].to_numpy()-PRED['OBS_METEO'][:,:,j][sel].reshape(-1))))<0.02
 print('PASS serie NOB histórica')
# NOB con resultados A, B1 y diferencia pareada por instante.
NOB_PAIRED=pd.DataFrame({'timestamp':nob_timestamps,'daytime':mask_d[sel].reshape(-1),
 'daytime_cs20':mask_dc[sel].reshape(-1)})
for k,v in enumerate(VARS):
 NOB_PAIRED[f'observed_{v}']=truth[:,:,k][sel].reshape(-1)
 for s,alias in [('OBS_METEO','OBS'),('TRAIN_CLIM_METEO','CLIM')]:
  NOB_PAIRED[f'pred_{v}_{alias}']=PRED[s][:,:,k][sel].reshape(-1)
 NOB_PAIRED[f'paired_pred_change_{v}']=NOB_PAIRED[f'pred_{v}_CLIM']-NOB_PAIRED[f'pred_{v}_OBS']
assert not NOB_PAIRED.isna().any().any()
print('PASS NOB pareado:',NOB_PAIRED.shape)

## 7. Exportación, hashes y manifiesto reproducible
Los CSV y JSON del paquete constituyen la evidencia de ejecución. No representan una comprobación externa independiente del hardware de Colab; incluyen entorno y parámetros suficientes para auditarla.

In [ ]:
from datetime import datetime,timezone
import zipfile, platform
OUTDIR=ROOT/'R1_C04_TEST_METEO_CLIM_V01_OUTPUT'
OUTDIR.mkdir(exist_ok=True)
results={
 'R1_C04_Climatology_TRAIN_1998_2019.csv':PROFILE.reset_index(),
 'R1_C04_Input_Replacement_Audit.csv':change_per_col,
 'R1_C04_OVERLAP_Metrics.csv':ALL_METRICS,
 'R1_C04_BY_LEAD_Metrics.csv':LEAD_METRICS,
 'R1_C04_NOB_Metrics.csv':NOB_METRICS,
 'R1_C04_NOB_Energy.csv':ENERGY,
 'R1_C04_OVERLAP_Paired_Delta.csv':OVERLAP_DELTA,
 'R1_C04_BY_LEAD_Paired_Delta.csv':LEAD_DELTA,
 'R1_C04_NOB_Paired_Delta.csv':NOB_DELTA,
 'R1_C04_NOB_Paired_Predictions.csv':NOB_PAIRED,
}
for filename,frame in results.items():
 frame.to_csv(OUTDIR/filename,index=False,float_format='%.10g')
manifest={
 'protocol':'R1_C04_TEST_Meteorological_Climatology_Sensitivity_V01',
 'utc_generated':datetime.now(timezone.utc).isoformat(),
 'source_code':'R1_C02_TEST_Benchmarks_Comparative_V01.ipynb + canonical 2050 cell 3 climatology algorithm',
 'climatology_training_interval':'1998-01-01 to 2019-12-31 23:30:00',
 'climatology_year_count':22,'climatology_profile_rows':int(len(PROFILE)),
 'leap_day_policy':'2024-02-29 -> per-slot arithmetic mean of 02-28 and 03-01 TRAIN median profiles',
 'scenario_A':'TEST future meteorology observed retrospectively',
 'scenario_B1':'TRAIN-only medians of 7 scalar meteorological variables plus median sine/cosine of wind direction',
 'changed_future_feature_names':METEO_FUT,
 'unchanged_future_feature_names':KEEP_FUT,
 'history_unchanged':True,'geometry_temporal_clearsky_unchanged':True,
 'transformers_frozen':True,'model_frozen':True,'training_performed':False,
 'test_windows':int(N),'horizon':int(cfg.H),'stride':int(cfg.stride),'nob_timestamps':int(len(nob_timestamps)),
 'daytime_definition_canonical':'cosz_plus > 0.05',
 'daytime_cs20_definition_diagnostic':'cosz_plus > 0.05 AND GHI_cs > 20',
 'different_mask_positions':int(np.sum(mask_d!=mask_dc)),
 'canonical_metric_max_abs_deltas':CANON_DELTAS,
 'limitations':['Not an operational forecasting comparison','Context historical observations retained in both scenarios','No recursive synthetic feedback','Change of exogenous information regime may involve distribution shift'],
 'input_files':{p.name:{'sha256':sha(p),'size_bytes':p.stat().st_size} for p in [DATA,MODEL,SC_H,SC_F,META,CANON_METRICS]+([CANON_NOB] if CANON_NOB else [])},
 'environment':{'python':platform.python_version(),'tensorflow':tf.__version__,'numpy':np.__version__,'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__,'batch_size':BATCH},
 'output_files':{name:{'sha256':sha(OUTDIR/name),'size_bytes':(OUTDIR/name).stat().st_size,'rows':len(df0)} for name,df0 in results.items()},
}
manifest_path=OUTDIR/'R1_C04_Manifest_V01.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
zip_path=ROOT/'R1_C04_TEST_METEOROLOGICAL_CLIMATOLOGY_SENSITIVITY_V01.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in sorted(OUTDIR.iterdir()):z.write(p,arcname=p.name)
with zipfile.ZipFile(zip_path) as z:assert z.testzip() is None
print('ZIP:',zip_path,'size:',zip_path.stat().st_size,'archivos:',len(results)+1)
print('Enviar este ZIP para auditoría científica de R1-4.')